# Agentic RAG and tool calling 

In [ ]:
# We'll use a typical openAI sdk, so lets import that first

import openai from openAI

openai_client = openAI()

### Documents loading

In [ ]:
# We'll have to bring our documents to use in this example from gitsource

from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

# we can parse and chunk straight from this package

### Create search index

In [ ]:
# Import minsearch and create an AppendableIndex (which allows adding new documents later):

from minisearch import AppendableIndex

# Create the index and fit it with our documents:

index = AppendableIndex(
    text_fields=["title", "description", "content"]
    keywords=["filename"]
)

index.fit(chunked_docs)

### Traditional RAG

In [ ]:
# Lets create our traditional rag, as we previously already created

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

# Now lets add in the rag prompt template

import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [ ]:
# Create a prompt to add to the RAG

question = "How do I create instruczions in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [ ]:
# Grab the responses from the RAG

messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [ ]:
# Get the answer from the RAG

print(response.output_text)

# We should now expect a bad response from the RAG because it does not
# handle typos well, its simple.

### Making It Agentic